# 02 — ONNX Runtime

Converts PP-LCNet_x1_0_doc_ori from PaddlePaddle to ONNX, then benchmarks with `onnxruntime`.

**Prerequisites:** Run `01_paddle_baseline.ipynb` first (needs the paddle model in `models/paddle/`).

In [ ]:
import sys
sys.path.insert(0, '..')

import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
from tqdm import tqdm

from src.preprocess import load_and_preprocess

MODEL_DIR = Path('../models')
ONNX_PATH = MODEL_DIR / 'model.onnx'
DATASET_CSV = Path('../data/dataset.csv')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f'OnnxRuntime version: {ort.__version__}')

In [ ]:
# --- Convert PaddlePaddle model to ONNX ---
# Run once. Produces models/model.onnx
import subprocess

if not ONNX_PATH.exists():
    result = subprocess.run([
        'paddle2onnx',
        '--model_dir', str(MODEL_DIR / 'paddle'),
        '--model_filename', 'inference.pdmodel',
        '--params_filename', 'inference.pdiparams',
        '--save_file', str(ONNX_PATH),
        '--opset_version', '11',
        '--enable_onnx_checker', 'True',
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
    else:
        print(f'Converted: {ONNX_PATH}')
else:
    print(f'ONNX model already exists: {ONNX_PATH}')

In [ ]:
# Inspect model I/O
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 4
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=sess_options,
    providers=['CPUExecutionProvider'],
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f'Input:  {input_name}  shape={session.get_inputs()[0].shape}')
print(f'Output: {output_name} shape={session.get_outputs()[0].shape}')

In [ ]:
def predict(image_path: str) -> tuple[int, float]:
    tensor = load_and_preprocess(image_path)
    outputs = session.run([output_name], {input_name: tensor})
    logits = outputs[0]
    pred_label = int(np.argmax(logits, axis=1)[0])
    confidence = float(np.max(logits))
    return pred_label, confidence

df = pd.read_csv(DATASET_CSV)

predictions, confidences, latencies = [], [], []

for row in tqdm(df.itertuples(), total=len(df), desc='ONNX Runtime inference'):
    t0 = time.perf_counter()
    pred, conf = predict(row.image_path)
    latencies.append((time.perf_counter() - t0) * 1000)
    predictions.append(pred)
    confidences.append(conf)

df['onnx_pred'] = predictions
df['onnx_conf'] = confidences
df['onnx_latency_ms'] = latencies
df.to_csv(RESULTS_DIR / 'onnx_results.csv', index=False)
print('Saved results/onnx_results.csv')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

accuracy = (df['onnx_pred'] == df['label']).mean()
avg_latency = df['onnx_latency_ms'].mean()
throughput = 1000 / avg_latency

print(f'Accuracy:         {accuracy:.4f}')
print(f'Avg latency:      {avg_latency:.2f} ms')
print(f'Throughput:       {throughput:.1f} img/s')
print()
print(classification_report(df['label'], df['onnx_pred'],
                             target_names=['0deg','90deg','180deg','270deg']))

cm = confusion_matrix(df['label'], df['onnx_pred'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            xticklabels=['0','90','180','270'],
            yticklabels=['0','90','180','270'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('ONNX Runtime — Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'onnx_confusion_matrix.png', dpi=150)
plt.show()